# Assignment 27- Chatbot with conversation history using Langchain

### Groq LLM - Groq model = llama-3.1-8b-instant. 

In [ ]:
from dotenv import load_dotenv

from langchain_groq import ChatGroq
from langchain_community.document_loaders import WebBaseLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from langchain_ollama import OllamaEmbeddings

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

# PART 1 — Messages & Message Placeholders
# Task 1: Understanding Chat Messages (Conceptual)

In [ ]:
# Answer briefly:
# 1. What are SystemMessage, HumanMessage, and AIMessage?

"""SystemMessage sets the overall behavior, rules, or instructions for the AI, 
HumanMessage represents the user's input or question, 
and AIMessage represents the assistant's previous response in the conversation."""

# 2. Why message-based prompting is better than single prompts?
"""Message-based prompting is better than single prompts because it preserves conversation 
structure. It help to separates roles clearly and supports chat history. Responses are
more context-aware and accurate."""


# Task 2: MessagePlaceholder Usage

In [ ]:

from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.messages import HumanMessage, AIMessage
from langchain_groq import ChatGroq
from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv()

# Initialize Groq chat model
llm = ChatGroq(model="llama-3.1-8b-instant")

# Create a chat prompt template with:
# A system instruction
# A placeholder for dynamic chat history
# A human input placeholder
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant."),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "{input}")
])

# Initialize an empty list to store conversation history
chat_history = []

# -------------------- Turn 1 --------------------

# First user message
user_input = "What is Python?"

# Inject chat history and current user input into the prompt
formatted_prompt = prompt.invoke({
    "chat_history": chat_history,
    "input": user_input
})

# Send the formatted prompt to the Groq model
response = llm.invoke(formatted_prompt)

# Print conversation
print("User:", user_input)
print("AI:", response.content)

# Save current turn into chat history
chat_history.append(HumanMessage(content=user_input))
chat_history.append(AIMessage(content=response.content))

# -------------------- Turn 2 --------------------

# Second user message
user_input = "Can you give me an example?"

# Inject updated chat history and current input
formatted_prompt = prompt.invoke({
    "chat_history": chat_history,
    "input": user_input
})

# Get model response
response = llm.invoke(formatted_prompt)

# Print conversation
print("User:", user_input)
print("AI:", response.content)

# Update chat history
chat_history.append(HumanMessage(content=user_input))
chat_history.append(AIMessage(content=response.content))

# -------------------- Turn 3 --------------------

# Third user message
user_input = "What about tuples?"

# Inject full conversation history and current user input
formatted_prompt = prompt.invoke({
    "chat_history": chat_history,
    "input": user_input
})

# Get model response
response = llm.invoke(formatted_prompt)

# Print conversation
print("User:", user_input)
print("AI:", response.content)

# Update chat history again
chat_history.append(HumanMessage(content=user_input))
chat_history.append(AIMessage(content=response.content))


# PART 2 — Conversation History Management
# Task 3- Basic Message History

In [ ]:

from langchain_core.messages import HumanMessage, AIMessage, SystemMessage
from langchain_groq import ChatGroq
from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv()

# Initialize Groq model
llm = ChatGroq(model="llama-3.1-8b-instant")

# Store conversation history in a list and begin with a system instruction
chat_history = [
    SystemMessage(content="You are a helpful assistant.")
]

# -------------------- First Interaction --------------------

# User sends first message
user_input = "What is Python?"

# Append user message to history
chat_history.append(HumanMessage(content=user_input))

# Pass full history to the LLM
response = llm.invoke(chat_history)

# Print AI response
print("User:", user_input)
print("AI:", response.content)

# Append AI response to history
chat_history.append(AIMessage(content=response.content))

# -------------------- Second Interaction --------------------

# User sends follow-up message
user_input = "Can you give me an example?"

# Append user message to history
chat_history.append(HumanMessage(content=user_input))

# Pass full updated history to the LLM
response = llm.invoke(chat_history)

# Print AI response
print("User:", user_input)
print("AI:", response.content)

# Append AI response to history
chat_history.append(AIMessage(content=response.content))

# -------------------- Third Interaction --------------------

# User sends another follow-up message
user_input = "What about tuples?"

# Append user message to history
chat_history.append(HumanMessage(content=user_input))

# Pass full conversation history again
response = llm.invoke(chat_history)

# Print AI response
print("User:", user_input)
print("AI:", response.content)

# Append AI response to history
chat_history.append(AIMessage(content=response.content))

# Print full stored message history
print("\nFull Chat History:")
for message in chat_history:
    print(type(message).__name__, ":", message.content)


# Task 4: Trimming Chat History

In [19]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.messages import HumanMessage, AIMessage
from langchain_groq import ChatGroq
from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv()

# Initialize Groq model
llm = ChatGroq(model="llama-3.1-8b-instant")

# Create chat prompt template with system message, chat history, and current user input
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant."),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "{input}")
])

# Store conversation history in a list
chat_history = []

# Function to trim old messages if history exceeds the allowed number of messages
def trim_chat_history(history, max_messages=4):
    """
    Keep only the most recent messages in chat history.
    Older messages are removed when the limit is exceeded.
    """
    if len(history) > max_messages:
        history = history[-max_messages:]
    return history

# -------------------- Turn 1 --------------------
user_input = "What is Python?"

# Trim history before sending it to the model
chat_history = trim_chat_history(chat_history, max_messages=4)

# Inject trimmed history and current input into the prompt
formatted_prompt = prompt.invoke({
    "chat_history": chat_history,
    "input": user_input
})

# Get model response
response = llm.invoke(formatted_prompt)

# Print conversation
print("User:", user_input)
print("AI:", response.content)

# Append current interaction to history
chat_history.append(HumanMessage(content=user_input))
chat_history.append(AIMessage(content=response.content))

# -------------------- Turn 2 --------------------
user_input = "Can you give me an example?"

# Trim history before next request
chat_history = trim_chat_history(chat_history, max_messages=4)

# Inject trimmed history and current input
formatted_prompt = prompt.invoke({
    "chat_history": chat_history,
    "input": user_input
})

# Get model response
response = llm.invoke(formatted_prompt)

# Print conversation
print("User:", user_input)
print("AI:", response.content)

# Append current interaction to history
chat_history.append(HumanMessage(content=user_input))
chat_history.append(AIMessage(content=response.content))

# -------------------- Turn 3 --------------------
user_input = "What about tuples?"

# Trim history before next request
chat_history = trim_chat_history(chat_history, max_messages=4)

# Inject trimmed history and current input
formatted_prompt = prompt.invoke({
    "chat_history": chat_history,
    "input": user_input
})

# Get model response
response = llm.invoke(formatted_prompt)

# Print conversation
print("User:", user_input)
print("AI:", response.content)

# Append current interaction to history
chat_history.append(HumanMessage(content=user_input))
chat_history.append(AIMessage(content=response.content))

# -------------------- Turn 4 --------------------
user_input = "What is the difference between list and tuple?"

# Trim older history if limit is exceeded
chat_history = trim_chat_history(chat_history, max_messages=4)

# Inject trimmed history and current input
formatted_prompt = prompt.invoke({
    "chat_history": chat_history,
    "input": user_input
})

# Get model response
response = llm.invoke(formatted_prompt)

# Print conversation
print("User:", user_input)
print("AI:", response.content)

# Append current interaction to history
chat_history.append(HumanMessage(content=user_input))
chat_history.append(AIMessage(content=response.content))

# Print final trimmed chat history
print("\nFinal Chat History:")
for message in chat_history:
    print(type(message).__name__, ":", message.content)

User: What is Python?
AI: **What is Python?**

Python is a high-level, interpreted programming language that is widely used for various purposes such as:

1. **Web Development**: Python can be used to build web applications using popular frameworks like Django and Flask.
2. **Data Analysis and Science**: Python is a popular choice for data analysis, machine learning, and scientific computing due to its extensive libraries such as NumPy, pandas, and scikit-learn.
3. **Automation**: Python can be used to automate tasks by interacting with operating systems and other applications.
4. **Scripting**: Python can be used to write scripts that perform specific tasks.
5. **Game Development**: Python can be used to build games using libraries like Pygame and Panda3D.

**Key Features of Python**

1. **Easy to Learn**: Python has a simple syntax and is easy to read and write.
2. **High-Level Language**: Python is a high-level language, meaning it abstracts away many low-level details, allowing dev

# PART 3 — Q&A Chatbot with Message History
# Task 5: Build Q&A Chatbot

In [ ]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.messages import HumanMessage, AIMessage
from langchain_groq import ChatGroq
from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv()

# Initialize Groq chat model
llm = ChatGroq(model="llama-3.1-8b-instant")

# Create a chat prompt template with:
# 1. A system instruction
# 2. A placeholder for previous conversation history
# 3. A placeholder for the current user question
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful Q&A assistant. Use previous conversation history to answer follow-up questions clearly and accurately."),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "{input}")
])

# Initialize an empty list to store conversation history
chat_history = []

# Define a function to handle one chatbot interaction
def ask_chatbot(user_input):
    global chat_history

    # Inject chat history and current user question into the prompt
    formatted_prompt = prompt.invoke({
        "chat_history": chat_history,
        "input": user_input
    })

    # Send the formatted prompt to the model
    response = llm.invoke(formatted_prompt)

    # Print user question and AI response
    print("User:", user_input)
    print("AI:", response.content)
    print("-" * 80)

    # Save current conversation turn in history
    chat_history.append(HumanMessage(content=user_input))
    chat_history.append(AIMessage(content=response.content))

# Test the chatbot with follow-up questions
ask_chatbot("Explain Python lists")
ask_chatbot("Give an example")
ask_chatbot("What about tuples?")


User: Explain Python lists
AI: **Python Lists**

In Python, a list is a collection of items that can be of any data type, including strings, integers, floats, and other lists. Lists are denoted by square brackets `[]` and are ordered, meaning that items have a specific position in the list.

**Creating a List**
------------------

You can create a list in several ways:

```python
# Method 1: Using square brackets []
my_list = [1, 2, 3, 4, 5]

# Method 2: Using the list() function
my_list = list([1, 2, 3, 4, 5])

# Method 3: Using the constructor of a list
my_list = list()
my_list.append(1)
my_list.append(2)
my_list.append(3)
print(my_list)  # Output: [1, 2, 3]
```

**Indexing and Slicing**
----------------------

Lists are indexed, meaning that each item has a specific position in the list. You can access an item by its index using square brackets `[]`.

```python
my_list = [1, 2, 3, 4, 5]
print(my_list[0])  # Output: 1
print(my_list[4])  # Output: 5
```

You can also slice a list to e

# Mini Project: Stateful Chatbot
# Task 6: Build Stateful Chatbot Application


In [ ]:


# Created Streamlit app.py

# - Streamlit UI with chat interface)

# Task 7: Observations & Insight

In [ ]:
# Write short answers:
# 1. Why chat history is important
"""Chat history helps the chatbot remember previous messages, understand follow-up questions, 
maintain context, and provide more relevant and natural responses.
"""
# 2. Trade-offs between long memory and performance
""" Longer memory improves context and continuity, but it increases token usage, 
slows response time, raises cost, and may reduce efficiency if too much irrelevant history is included.
"""

# 3. When to summarize vs trim history
""" Summarize history when older conversation is still important but too long to keep fully, and trim history 
when old messages are no longer useful and only recent context is needed."""

# 4. Difference between message placeholders and memory
"""Message placeholders are used in prompts to dynamically insert chat history, while memory is the 
mechanism that stores, manages, and retrieves past conversation for use in the chatbot. """
